# Pipeline 02: Agente RAG y LLM

Este notebook toma las predicciones generadas (predicciones_riesgo) y los documentos estructurados para indexarlos en PGVector y permitir consultas en lenguaje natural.

In [1]:
import sys
import os
import importlib
from pathlib import Path
from dotenv import load_dotenv

current_dir = Path.cwd()
ROOT_DIR = current_dir if (current_dir / "src").exists() else current_dir.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

load_dotenv(ROOT_DIR / ".env", override=True)

from src.config import settings
from src.db_utils import create_supabase_engine
import pandas as pd

engine = create_supabase_engine(settings.DATABASE_URL)

## 1. Poblar base vectorial RAG

Indexa en PGVector el resumen del portafolio, políticas de negocio y fichas de créditos.

In [2]:
import scripts.populate_rag as _populate_rag
importlib.reload(_populate_rag)
run_populate_rag = _populate_rag.main

chunks_indexados = run_populate_rag(reset=True, include_fichas=True)
print(f"Chunks indexados en PGVector: {chunks_indexados}")

rag_counts = pd.read_sql("""
SELECT
    (SELECT COUNT(*) FROM langchain_pg_collection) AS colecciones,
    (SELECT COUNT(*) FROM langchain_pg_embedding) AS embeddings;
""", engine)
display(rag_counts)

INFO - === Iniciando población del RAG Knowledge Base ===
INFO - Conectando a PostgreSQL y cargando predicciones_riesgo...
INFO -   → 1527 filas cargadas
INFO - Construyendo resumen del portafolio...
INFO -   → 1 documento(s) de resumen
INFO - Construyendo documentos de políticas de negocio...
INFO -   → 4 documentos de política
INFO - Construyendo fichas de perfil por crédito...
INFO -   → 1527 fichas de cliente/crédito
INFO - Total documentos antes de chunking: 1532
INFO - Total chunks a indexar: 1538
INFO - Cargando modelo de embeddings 'intfloat/multilingual-e5-small'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

INFO - Cargando documentos en PGVector (puede tardar varios minutos)...
c:\Daniel\Mio\project-data-scientist\.venv\Lib\site-packages\langchain_community\vectorstores\pgvector.py:490: LangChainPendingDeprecationWarning: Please use JSONB instead of JSON for metadata. This change will allow for more efficient querying that involves filtering based on metadata. Please note that filtering operators have been changed when using JSONB metadata to be prefixed with a $ sign to avoid name collisions with columns. If you're using an existing database, you will need to create a db migration for your metadata column to be JSONB and update your queries to use the new operators. 
  store = cls(
INFO - RAG Knowledge Base poblado con 1538 chunks.
INFO - Verifica con: SELECT COUNT(*) FROM langchain_pg_embedding;


Chunks indexados en PGVector: 1538


,colecciones,embeddings
0,1,1538


## 2. Preguntar al agente RAG + LLM

El agente recupera contexto desde PGVector y llama al LLM para responder con base en los datos indexados.

In [3]:
from src.rag_agent import RAGAgentPipeline

agent = RAGAgentPipeline(
    db_connection=settings.DATABASE_URL,
    nvidia_api_key=settings.NVIDIA_API_KEY,
)

def preguntar_tumipay(pregunta: str) -> str:
    respuesta = "".join(agent.stream_query(pregunta)).strip()
    print(f"Pregunta:\n{pregunta}\n\nRespuesta del agente:\n{respuesta}")
    return respuesta

pregunta = "¿Cuál es la tasa de mora del portafolio y qué acciones recomiendas para los segmentos de mayor riesgo?"
respuesta = preguntar_tumipay(pregunta)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

c:\Daniel\Mio\project-data-scientist\src\rag_agent.py:45: LangChainPendingDeprecationWarning: This class is pending deprecation and may be removed in a future version. You can swap to using the `PGVector` implementation in `langchain_postgres`. Please read the guidelines in the doc-string of this class to follow prior to migrating as there are some differences between the implementations. See <https://github.com/langchain-ai/langchain-postgres> for details about the new implementation.
  self.vector_store = PGVector(
c:\Daniel\Mio\project-data-scientist\src\rag_agent.py:45: LangChainPendingDeprecationWarning: Please use JSONB instead of JSON for metadata. This change will allow for more efficient querying that involves filtering based on metadata. Please note that filtering operators have been changed when using JSONB metadata to be prefixed with a $ sign to avoid name collisions with columns. If you're using an existing database, you will need to create a db migration for your metadat

Pregunta:
¿Cuál es la tasa de mora del portafolio y qué acciones recomiendas para los segmentos de mayor riesgo?

Respuesta del agente:
Según el contexto proporcionado, la tasa de mora del portafolio es del 27.2%, lo que se traduce en 416 clientes morosos identificados sobre un total de 1,527 créditos en portafolio.

Para los segmentos de mayor riesgo, recomiendo las siguientes acciones:

1. **Revisión de la relación cuota/ingreso**: Es importante revisar la relación cuota/ingreso de los clientes morosos para determinar si es una cuestión de capacidad de pago o de gestión financiera. Si la relación es alta, es posible que los clientes estén experimentando dificultades para pagar sus cuotas.
2. **Oferta de refinanciación**: Ofrecer refinanciación a los clientes morosos puede ser una forma de ayudarlos a pagar sus cuotas sin que se vean afectados por la mora. Es importante que la refinanciación sea ofrecida en condiciones razonables y que no se aumente el monto total del crédito.
3. **Ge